# What a cold build costs

`pipeline/build_places.py` is the only thing here that calls an LLM, and only for rows
whose hours the regex parser cannot read or that carry seasonal notes. This notebook
prices a build starting from an empty `data/hours_llm_cache.json`.

It imports `route_hours` and `cache_key` from the build itself, so the call count is the
real one rather than a guess. With the cache committed, a normal rebuild costs nothing.

In [1]:
import json
import sys

sys.path.insert(0, "pipeline")  # pipeline/ imports are flat

from build_places import SOURCE, route_hours
from llm_hours_parser import PROMPT, PlaceHours, cache_key, get_client, load_hours_cache

# $ per 1M (input, output) tokens, short context.
# developers.openai.com/api/docs/pricing, Sep 2026. Sol's rate is promotional to Nov 21.
PRICES = {
    "gpt-5.6-sol": (4.00, 20.00),  # what llm_hours_parser.py asks for today
    "gpt-5.6-luna": (0.20, 1.20),
    "gpt-4o": (2.50, 10.00),
    "gpt-4o-mini": (0.15, 0.60),
}

SCHEMA = json.dumps(PlaceHours.model_json_schema())  # rides along on every call
CHAT_OVERHEAD = 10  # role and formatting tokens the API wraps the message in

# 4 chars per token under-counts this workload: prompt and answer are mostly quotes,
# colons and digits, which tokenize worse than prose. These two ratios come from the
# real call in the last cell -- rerun it to re-measure.
IN_CALIBRATION = 1116 / 1013
OUT_CALIBRATION = 285 / 148


def tokens(text):
    return round(len(text) / 4)


def prompt_for(hours, notes):
    return PROMPT.format(hours=hours if hours else "(none)", seasonal_notes=notes or "")


def as_model_output(entry):
    """A cached answer back in the shape the model emitted it, so it can be counted."""
    return json.dumps(
        {
            "open_days": [
                {"day": day, "hours": day_hours["hours"], "weeks": day_hours["weeks"]}
                for day, day_hours in entry["open_days"].items()
            ],
            "open_months": entry["open_months"],
        },
        separators=(",", ":"),
    )

In [2]:
# Distinct (hours, notes) pairs only: rows sharing a string cost one call, same as the build.
rows = json.loads(SOURCE.read_text(encoding="utf-8"))
pending = {}
for row in rows:
    row = {**row, "posted_hours": row["hours"]}  # clean_frame renames it
    if route_hours(row)[0] == "llm":
        key = cache_key(row["posted_hours"], row["seasonal_notes"])
        pending[key] = (row["posted_hours"], row["seasonal_notes"])

cache = load_hours_cache()
cached = pending.keys() & cache.keys()
print(f"{len(rows)} places -> {len(pending)} LLM calls on a cold build")
print(f"{len(cached)} of them answered in the cache already, so a rebuild today sends none")

103 places -> 16 LLM calls on a cold build
16 of them answered in the cache already, so a rebuild today sends none


In [3]:
in_tokens = round(
    sum(tokens(prompt_for(*pair)) + tokens(SCHEMA) + CHAT_OVERHEAD for pair in pending.values())
    * IN_CALIBRATION
)
answers = [tokens(as_model_output(cache[key])) for key in cached]
out_tokens = round(sum(answers) / len(answers) * len(pending) * OUT_CALIBRATION)


def report(in_tokens, out_tokens):
    calls = len(pending)
    print(f"per call: {round(in_tokens / calls):,} in, {round(out_tokens / calls):,} out")
    print(f"totals:   {in_tokens:,} in, {out_tokens:,} out\n")
    for model, (price_in, price_out) in PRICES.items():
        cost = (in_tokens * price_in + out_tokens * price_out) / 1_000_000
        print(f"  {model:<13} ${cost:.4f}")


report(in_tokens, out_tokens)

per call: 1,114 in, 211 out
totals:   17,823 in, 3,374 out

  gpt-5.6-sol   $0.1388
  gpt-5.6-luna  $0.0076
  gpt-4o        $0.0783
  gpt-4o-mini   $0.0047


## Re-measuring

Everything above is arithmetic on the cached answers. The cell below spends one real call
(about half a cent) to check it, and recomputes the table from the billed counts.

The probe answered `reasoning_tokens: 0`, which is why no reasoning allowance is priced in
for the gpt-5.6 models: structured hours parsing at the default effort does not reason.
Watch that number if the prompt or the schema grows -- reasoning is billed at the output
rate, so it would land hardest on Sol.

In [ ]:
key, (hours, notes) = next(iter(pending.items()))
usage = (
    get_client()
    .chat.completions.parse(
        model="gpt-5.6-sol",
        messages=[{"role": "user", "content": prompt_for(hours, notes)}],
        response_format=PlaceHours,
    )
    .usage
)

estimated_in = tokens(prompt_for(hours, notes)) + tokens(SCHEMA) + CHAT_OVERHEAD
estimated_out = tokens(as_model_output(cache[key]))
print(f"live call on {hours!r}")
print(f"  input      {usage.prompt_tokens} billed vs {estimated_in} estimated")
print(f"  completion {usage.completion_tokens} billed vs {estimated_out} estimated")
print(f"  reasoning  {usage.completion_tokens_details.reasoning_tokens}\n")

# Drop the stored calibration, apply what this call just measured.
report(
    round(in_tokens / IN_CALIBRATION * usage.prompt_tokens / estimated_in),
    round(out_tokens / OUT_CALIBRATION * usage.completion_tokens / estimated_out),
)